# Module 5 · Project — Knowledge Assistant v2

**From 0 to Agentic AI — DataHack Summit 2026**

🧩 **v2 — real tools.** We upgrade the assistant from toy tools to real external systems:
a custom **web search** (wrapping the Seltz API), and two **action** tools that draft a
**Slack** message and a **GitHub** issue. Same graph as v1 — richer hands.

> 📝 **Your turn.** Cells marked **TODO** need implementing. Run each section as you go. Stuck? Peek at the `_SOLUTION` notebook.

### What you'll build
1. `web_search` — wrap the **Seltz** client as a `@tool` (connecting an external API)
2. `draft_slack_message` — build a Slack payload (**dry-run**, doesn't send)
3. `draft_github_issue` — build a GitHub issue payload (**dry-run**)
4. Bind all three into the v1 graph → **v2**, and drive it

### 🏗️ Architecture
```
  user ──▶ agent ──(tool call?)──▶ tools: web_search | slack | github
            ▲                              │
            └────────── observation ◀──────┘
```
Same reason→act→observe loop from Module 4; we're only changing the tools.

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml (the single source of truth).
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" \
               "langgraph>=1.0,<2" "seltz>=1.5.0"

In [ ]:
import os
from getpass import getpass

# Local: load keys from src/.env (walks up to find it). Colab: prompts for missing keys.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY", "SELTZ_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Tool 1 · Web search (wrapping the Seltz API)

This is the heart of *connecting an external API*: the Seltz client is a plain Python object;
we wrap a call to it in a `@tool` with a good docstring and a `try/except`.

**TODO:** finish `web_search` — call `client.search(query, max_results=...)`, then join the
returned `resp.documents` (each has `.url` and `.content`) into a readable string. Wrap it in
a `try/except` that returns a readable error.

In [ ]:
from langchain_core.tools import tool
from seltz import Seltz

_seltz = Seltz()   # reads SELTZ_API_KEY from the environment

In [ ]:
@tool
def web_search(query: str, max_results: int = 3) -> str:
    """Search the web for current, external information."""
    # TODO: call _seltz.search(query, max_results=max_results) in a try/except
    #       (return f'search error: {e}' on failure)
    # TODO: join resp.documents (each has .url and .content) into a readable string
    ...

In [ ]:
print(web_search.invoke({"query": "What is LangGraph?"})[:300])

---
## Tool 2 · Draft a Slack message (dry-run)

Our assistant can *act* — but safely. Rather than posting to Slack, this tool **returns the
message payload for review**. (Actually sending is a one-line `requests.post` to a Slack
webhook — left out so the workshop needs no tokens.)

**TODO:** return a JSON string with keys `action`, `channel`, `message`.

In [ ]:
import json

@tool
def draft_slack_message(channel: str, message: str) -> str:
    """Draft a Slack message to a channel for review (does NOT send)."""
    # TODO: return json.dumps({...}) with keys "action", "channel", "message"
    ...

---
## Tool 3 · Draft a GitHub issue (dry-run)

Same pattern — build the issue payload, don't create it.

**TODO:** return a JSON string with keys `action`, `repo`, `title`, `body`.

In [ ]:
@tool
def draft_github_issue(repo: str, title: str, body: str) -> str:
    """Draft a GitHub issue for a repo for review (does NOT create it). repo is 'owner/name'."""
    # TODO: return json.dumps({...}) with keys "action", "repo", "title", "body"
    ...

### Test the tools standalone
Always exercise a tool directly before handing it to the agent.

In [ ]:
print(draft_slack_message.invoke({"channel": "#billing", "message": "Deploy is failing."}))
print()
print(draft_github_issue.invoke(
    {"repo": "acme/api", "title": "Deploy fails", "body": "The 3pm deploy failed."}))

---
## Assemble v2

Same graph as v1 (Module 4) — we just pass the new tool list. This is why the graph
abstraction pays off: swapping capabilities doesn't touch the loop.

In [ ]:
from typing import Annotated, TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

tools = [web_search, draft_slack_message, draft_github_issue]

SYSTEM_PROMPT = (
    "You are the Knowledge Assistant for a software company. "
    "Answer questions using web search when useful. When the user asks to notify "
    "someone or file an issue, draft it with the appropriate tool and show them the draft."
)

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state: State):
    msgs = [("system", SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(msgs)]}

def should_continue(state: State) -> str:
    return "tools" if state["messages"][-1].tool_calls else "end"

builder = StateGraph(State)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
assistant = builder.compile()

display(Image(assistant.get_graph().draw_mermaid_png()))

---
## Run it

One question that needs **search**, and one that needs an **action** (a Slack draft).

In [ ]:
def ask(q):
    out = assistant.invoke({"messages": [("user", q)]})
    return out["messages"][-1].content

print(ask("What is the latest major version of LangGraph?"))

In [ ]:
out = assistant.invoke({"messages": [(
    "user", "The billing service deploy is failing — draft a Slack message to #billing.")]})
for m in out["messages"]:
    m.pretty_print()

---
## Key takeaways
- **Wrapping an API as a `@tool`** is the core move for connecting external systems.
- **Draft / dry-run** action tools let an agent *act* safely — no tokens, review before send.
- Good **docstrings + try/except** make the difference between a reliable tool and a flaky one.
- Adding tools **didn't change the graph** — the Module 4 loop just gained new hands.

➡️ **Next (Module 6):** give the assistant real internal knowledge — **RAG**, context
engineering, and MCP — so it can answer about *your* docs, not just the web.